# Data preparation - Cleaning and Feature Engineering
Created by Guillermo Arredondo Renero

Creation date: May 4, 2026  
Last updated: May 4, 2026

**Main objective**: This notebook provides the step by step data preparation based on the findings of the previous notebook. 

**Structure:**


In [1]:
# ---------------------------- Libraries
## Directories
import os
## Data manipulation
import pandas as pd
import numpy as np

## Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable

# Extras
import warnings
warnings.filterwarnings('ignore')

# Transformation

In [2]:
# ---------------------------- Directories
# Get main directory
CURRENT_DIR = os.getcwd()
MAIN_DIR = os.path.dirname(CURRENT_DIR)
DATA_DIR = os.path.join(MAIN_DIR, 'data', 'raw')
INTERIM_DIR = os.path.join(MAIN_DIR, 'data', 'interim')
os.makedirs(INTERIM_DIR, exist_ok=True)
PROCESSED_DIR = os.path.join(MAIN_DIR, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Data loading

In [3]:
df = pd.read_csv(os.path.join(DATA_DIR, 'credit_dataset.csv'))

## Data cleaning

### Inconsistent values and named variables

In [ ]:
df.rename(columns={'default payment next month': 'default_payment_next_month',
                   'PAY_0': 'PAY_1'}, 
          inplace=True)

# Filter data with undocumented values for Education and Marriage
## Nota: podría haberse considerado agrupar estas categorías dentro de una categoría general
## sin embargo, debido a la falta de documentación y la búsqueda de interpretabilidad en modelos créditicios, 
## se opta por eliminar estas filas para mantener la claridad en las categorías de estas variables.
df = df[(df['EDUCATION'].isin([1, 2, 3, 4])) & (df['MARRIAGE'].isin([1, 2, 3]))]

# Re-scale PAY_i variables
## Add 1 to all PAY_i columns to shift values from [-2, 8] to [-1, 9]
## Group -1 and 0 as "On time" (-1), and keep 1-9 as is (1-9)
pay_cols = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for col in pay_cols:
    df[col] = df[col] + 1
    df[col] = df[col].apply(lambda x: -1 if x <= 0 else x)

print("Show new dataset info:")
print(df.shape)
print(df.info())
print("\nShow new dataset description:")
df.describe()

Show new dataset info:
(29601, 24)
<class 'pandas.DataFrame'>
Index: 29601 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   LIMIT_BAL                   29601 non-null  int64
 1   SEX                         29601 non-null  int64
 2   EDUCATION                   29601 non-null  int64
 3   MARRIAGE                    29601 non-null  int64
 4   AGE                         29601 non-null  int64
 5   PAY_1                       29601 non-null  int64
 6   PAY_2                       29601 non-null  int64
 7   PAY_3                       29601 non-null  int64
 8   PAY_4                       29601 non-null  int64
 9   PAY_5                       29601 non-null  int64
 10  PAY_6                       29601 non-null  int64
 11  BILL_AMT1                   29601 non-null  int64
 12  BILL_AMT2                   29601 non-null  int64
 13  BILL_AMT3                   29601 non-null

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_payment_next_month
count,29601.000000,29601.000000,29601.000000,29601.000000,29601.000000,29601.000000,29601.000000,29601.000000,29601.000000,29601.000000,...,29601.000000,29601.000000,29601.000000,29601.000000,2.960100e+04,29601.000000,29601.000000,29601.000000,29601.000000,29601.000000
mean,167550.544914,1.603189,1.815479,1.555454,35.464072,0.794770,0.666329,0.638492,0.591939,0.550894,...,43122.554204,40235.545184,38858.449816,5649.560319,5.894788e+03,5198.415898,4828.659268,4795.032735,5181.326374,0.223134
std,129944.020953,0.489244,0.710399,0.518092,9.213243,1.339224,1.397190,1.389153,1.350751,1.309342,...,64196.383913,60699.344884,59519.893043,16568.264941,2.308919e+04,17580.914806,15711.057992,15244.217154,17657.260739,0.416355
min,10000.000000,1.000000,1.000000,1.000000,21.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2329.000000,1780.000000,1278.000000,1000.000000,8.250000e+02,390.000000,298.000000,259.000000,138.000000,0.000000
50%,140000.000000,2.000000,2.000000,2.000000,34.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,19005.000000,18091.000000,17118.000000,2100.000000,2.007000e+03,1800.000000,1500.000000,1500.000000,1500.000000,0.000000
75%,240000.000000,2.000000,2.000000,2.000000,41.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,54271.000000,50072.000000,49121.000000,5005.000000,5.000000e+03,4500.000000,4014.000000,4042.000000,4000.000000,0.000000
max,1000000.000000,2.000000,4.000000,3.000000,79.000000,9.000000,9.000000,9.000000,9.000000,9.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.000000,621000.000000,426529.000000,528666.000000,1.000000


Notamos que pese a la eliminación de algunos registros no se modifica el porcentaje de default

In [5]:
print("Data cleaned successfully. Saving interim dataset...")
df.to_csv(os.path.join(INTERIM_DIR, 'credit_dataset_cleaned.csv'), index=False)

Data cleaned successfully. Saving interim dataset...


## Feature engineering

### 